# PolicyRec Gemini RAG 시스템 (v1.2)

이 노트북은 모든 과정을 **Google Gemini**를 사용하여 처리합니다.
- **Embedding**: `gemini-embedding-001` (768차원)
- **Generation**: `gemini-2.5-flash-lite` (빠르고 효율적인 답변 생성)


In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai
from supabase import create_client, Client

# 1. 환경 설정 및 클라이언트 초기화
load_dotenv()

# Gemini API 설정
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# 답변 생성용 모델 설정 (gemini-2.5-flash-lite 사용)
model = genai.GenerativeModel('gemini-2.5-flash-lite')

# Supabase 설정
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

print("✅ Gemini 및 Supabase 초기화 완료")

✅ Gemini 및 Supabase 초기화 완료 (OpenAI 미사용)


In [2]:
# 2. 질문 임베딩 함수 (Gemini)
def get_query_embedding(text):
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        task_type="retrieval_query",
        output_dimensionality=768
    )
    return result['embedding']

# 3. Supabase 유사도 검색 함수
def search_similar_announcements(query_text, threshold=0.3, count=3):
    query_embedding = get_query_embedding(query_text)
    
    response = supabase.rpc(
        "match_announcements",
        {
            "query_embedding": query_embedding,
            "match_threshold": threshold,
            "match_count": count
        }
    ).execute()
    
    return response.data

# 4. Gemini 답변 생성 함수
def generate_answer(query_text, search_results):
    # 컨텍스트 구성
    context = ""
    for i, res in enumerate(search_results):
        context += f"[{i+1}] 제목: {res['title']}\n"
        context += f"분류: {res['category']} | 지역: {res['region']}\n"
        context += f"내용 요약: {res['content']}\n"
        context += f"상세 링크: {res['detail_url']}\n\n"
    
    prompt = f"""당신은 대한민국의 다양한 정부 혜택과 공고를 안내하는 전문 상담사입니다. 
제공된 [컨텍스트] 내용을 바탕으로 사용자의 질문에 친절하고 상세하게 답변하세요. 
답변 시 참고한 공고의 제목과 상세 링크를 반드시 포함하고, 만약 관련 있는 정보가 없다면 정중히 모른다고 답변하세요.

[사용자 질문]: {query_text}

[컨텍스트]:
{context}

위 내용을 바탕으로 사용자에게 최적의 정책을 추천하고 안내해 주세요."""

    # Gemini 답변 생성
    response = model.generate_content(prompt)
    return response.text

In [6]:
# 5. 실행부
def run_policy_rag(query):
    print(f"🔍 질문 분석 중: {query}")
    results = search_similar_announcements(query)
    
    if not results:
        return "죄송합니다. 관련된 공고 정보를 찾을 수 없습니다."
    
    print(f"✅ {len(results)}개의 관련 공고를 찾았습니다.")
    return generate_answer(query, results)

# 테스트 질문
user_query = "울산에서 일하는 중소기업 청년을 위한 혜택이 있을까?"
answer = run_policy_rag(user_query)
print("\n🤖 Gemini AI 상담사 답변:\n")
print(answer)

🔍 질문 분석 중: 울산에서 일하는 중소기업 청년을 위한 혜택이 있을까?
✅ 3개의 관련 공고를 찾았습니다.

🤖 Gemini AI 상담사 답변:

안녕하세요! 울산에서 일하시는 청년분들을 위한 혜택에 대해 문의주셨네요.

안타깝게도 제공된 [컨텍스트] 내용 중에는 **울산 지역의 중소기업에 재직 중인 청년분들에게 직접적으로 적용되는 혜택에 대한 정보가 포함되어 있지 않습니다.**

현재 제가 가지고 있는 정보는 다음과 같습니다.

*   **[1] 울산시 바이오 디지털 헬스케어 글로벌 진출 지원 (2026 BIO USA 참관) 사업:** 이 사업은 울산지역의 바이오‧디지털 헬스케어 분야의 **일반기업**을 대상으로 하고 있습니다.
    *   상세 링크: [https://www.k-startup.go.kr/web/contents/bizpbanc-ongoing.do?schM=view&pbancSn=177319](https://www.k-startup.go.kr/web/contents/bizpbanc-ongoing.do?schM=view&pbancSn=177319)

*   **[2] 서귀포시 청년농업인 영농정착 지원:** 이 사업은 제주특별자치도 서귀포시의 **청년농업인**을 대상으로 합니다.
    *   상세 링크: [https://www.nongupez.go.kr/nsm/main](https://www.nongupez.go.kr/nsm/main)

*   **[3] ‘빛나는 제주 청년희망 대출’ 이차보전:** 이 사업은 제주특별자치도에 거주하거나 근무하는 **도내 근로소득 청년**을 대상으로 합니다.
    *   상세 링크: [https://plus.gov.kr/portal/benefitV2/benefitTotalSrvcList/benefitSrvcDtl?svcSeq=2154016&bnefType=all&svcId=650000001131](https://plus.gov.kr/portal/benefitV2/benefitTotalSrvcList/benef